Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0

# Sign in to your space

Omni lives outside the AWS Management Console, at your domain's URL. The workshop domain has no identity
provider, so you sign in as your IAM role with a one-time deep link: a short-lived, single-use code that
establishes an Omni session for the role that minted it.

In [ ]:
%pip install --quiet "boto3==1.43.126" "botocore==1.43.126"
import boto3

assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 43, 100), (
    f"boto3 too old: {boto3.__version__} (need >=1.43.100 for the cloudwatchomni client)"
)
# If the kernel imported an older botocore before the pip install above, the new
# service models are not loaded yet. Fail here with an actionable message.
services = boto3.session.Session().get_available_services()
assert "cloudwatchomni" in services and "observabilityadmin" in services, (
    "This kernel has no cloudwatchomni client. Restart the kernel (Kernel > Restart) and run this cell again."
)
print(f"boto3 {boto3.__version__} OK - cloudwatchomni and observabilityadmin clients available")

In [ ]:
import json
import os
import pathlib
import time

import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError("No AWS region configured. Set AWS_REGION or run `aws configure set region <region>`.")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# The CLI pages save identifiers as `export KEY=VALUE` lines in this file, so a
# participant can switch between the CLI and notebook paths at any step.
STATE_FILE = pathlib.Path.home() / ".workshop-omni.env"


def load_state():
    state = {}
    if STATE_FILE.exists():
        for line in STATE_FILE.read_text().splitlines():
            if line.startswith("export ") and "=" in line:
                key, value = line[len("export "):].split("=", 1)
                state[key] = value.strip()
    return state


def save_state(**values):
    state = load_state()
    state.update({k: v for k, v in values.items() if v})
    STATE_FILE.write_text("".join(f"export {k}={v}\n" for k, v in state.items()))
    return state


omni = boto3.client("cloudwatchomni", region_name=REGION)
obs_admin = boto3.client("observabilityadmin", region_name=REGION)
logs = boto3.client("logs", region_name=REGION)
cfn = boto3.client("cloudformation", region_name=REGION)
print(f"Region {REGION}, account ...{ACCOUNT_ID[-4:]}")

## Step 1: Mint a deep link (valid 10 minutes, single use)

In [ ]:
state = load_state()
DOMAIN_ID = state.get("OMNI_DOMAIN_ID") or omni.list_domains()["items"][0]["domainId"]
domain_name = omni.get_domain(domainId=DOMAIN_ID)["domain"]["name"]
# The sign-in host is the domain NAME, not the domainEndpointUrl (an internal d-... host).
domain_url = f"https://{domain_name}.cloudwatch-omni.global.app.aws"
link = omni.create_one_time_deep_link_code(domainId=DOMAIN_ID, ttlSeconds=600)
# deepLinkUrl is the code's path on the domain (/auth/code?code=...), not a full URL.
print("Open this URL in a new browser tab (it works once):")
print(domain_url + link["deepLinkUrl"])
print(f"expires at {link['expiresAt']}")

## Step 2: Open the space

The link opens **workshop-space** directly (the domain has one space; the name sits bottom-left, above
**Settings**, and is the switcher if you ever have more). The home page lists every view in three groups,
the map for the rest of the module: **Application performance monitoring** (Application map, Services,
Traces), **Analytics and dashboards** (Explore logs, Explore metrics, Alerts, Investigations, Dashboards,
Markdown) and **Agent observability** (Agent overview, Evaluation dashboard, Evaluators, Online evaluations,
Agent traces, Agent sessions, Agent topology, Datasets, Prompt playground).

Open **Settings**, then **Manage permissions**: four principals. This kernel's identity, the Code Editor
instance role, holds `code-editor-space-admin` (*Space admin*, *Customer managed*, created by the
`code-editor` stack); `WSParticipantRole` holds `participant-space-admin` (same level, created by
`workshop-agentcore-stack`); `WSSystemRole`, which ran CloudFormation for Workshop Studio and therefore
created the space, holds the system-managed `OmniDefaults-Space-Admin`; and the space's own agent profile
holds a custom `OmniDefaults-AsyncAccessProfile-AgentCanAssume` grant. In your own account the creator is the
identity that ran `./scripts/self-service-deploy.sh`, so it is the one holding `OmniDefaults-Space-Admin`;
there is no `WSSystemRole` row, no `participant-space-admin` row unless you deployed with
`OMNI_CONSOLE_PRINCIPAL_ARN` set to another identity, and this kernel's `code-editor-space-admin` row is the
same.
Grants only add (access is the union of every grant a principal holds), and the space operator role is the
ceiling for everything in the space.

A second click on the same link fails with a 400: the code is deleted when redeemed. Re-run Step 1 for a
new one. The CloudWatch console's Omni page (**Launch Omni → Sign in**) signs `WSParticipantRole` in the
same way without a link.

Continue with `04-platform-signals.ipynb` (Tracks 2 and 3) or `05-agent-traces.ipynb` (Track 1).